# 🚗 Semantic Segmentation for Autonomous Driving

## How does a self-driving car understand the road — pixel by pixel?

Object detection tells us **what** is in a scene. But autonomous driving also needs to understand **where each part of the scene is**.

That is the role of **Semantic Segmentation**.

---
## 📖 Read More & See It in Action

Want to go deeper into semantic segmentation and its role in autonomous driving?

📝 **Read the full article on Medium:**
[Semantic Segmentation for Autonomous Driving — From Pixels to Scene Understanding](https://theautojourney.medium.com/from-pixels-to-driving-decisions-understanding-image-segmentation-for-autonomous-driving-dd54a7a8f4b9)

🎥 **See the segmentation pipeline in action:**

![Semantic Segmentation Demo](docs/segmentation.gif)

The demonstration uses **CARLA Simulator** to visualize how an autonomous-driving camera image can be transformed into a pixel-level semantic representation of the environment.

---

## 🧠 What is Semantic Segmentation?

Semantic segmentation assigns a **class label to every pixel** in an image.

Typical classes in a driving environment include:

- 🛣️ Road
- 🚗 Car
- 🚶 Pedestrian
- 🚦 Traffic Signal
- 🏙️ Sidewalk
- 🌳 Vegetation
- ☁️ Background

Unlike object detection, which typically produces bounding boxes, semantic segmentation produces a **dense semantic map** of the entire scene.

```text
Camera
   ↓
Neural Network
   ↓
Pixel Classification
   ↓
Semantic Scene Map
````

---

## 🚘 Why is it important for autonomous driving?

A vehicle needs more than knowing:

> "There is a car ahead."

It needs to understand the **spatial structure of the environment**.

For example:

* Which areas are drivable?
* Where does the road end?
* Where is the sidewalk?
* Which pixels belong to vehicles or pedestrians?
* Where are obstacles and other scene elements?

This spatial understanding can support downstream tasks such as:

* **Free-space estimation**
* **Path planning**
* **Obstacle understanding**
* **Scene understanding**
* **Drivable-area estimation**

---

## 🔬 Where does U-Net fit in?

One of the most influential segmentation architectures is **U-Net**.

Originally developed for biomedical image segmentation, U-Net uses an **encoder-decoder architecture**:

```text
Input Image
     ↓
   Encoder
     ↓
Feature Extraction
     ↓
   Decoder
     ↓
Segmentation Map
```

### Encoder

The encoder extracts increasingly complex features from the input image.

It learns to recognize patterns such as:

* Edges
* Shapes
* Textures
* Objects
* High-level semantic features

### Decoder

The decoder progressively reconstructs the spatial resolution of the image to produce **pixel-level predictions**.

### Skip Connections

A key feature of U-Net is its **skip connections**.

They connect corresponding layers of the encoder and decoder, helping preserve fine spatial information that can otherwise be lost during downsampling.

This is particularly useful for reconstructing:

* Object boundaries
* Thin structures
* Small objects
* Fine spatial details

---

## 📊 How do we evaluate Semantic Segmentation?

A common evaluation metric is **Intersection over Union (IoU)**.

The formula is:

$$
IoU = \frac{Intersection}{Union}
$$

IoU measures how much the predicted region overlaps with the ground-truth region.

### Why per-class IoU matters

We shouldn't look only at the overall score.

A model could perform very well on large classes such as:

* 🛣️ Road
* 🏙️ Building
* 🌳 Vegetation

while struggling with smaller classes such as:

* 🚶 Pedestrians
* 🚲 Bicycles
* 🚦 Traffic Signals

That's why **per-class IoU** is particularly valuable when evaluating segmentation models.

---

## 🏙️ Cityscapes

Benchmarks such as **Cityscapes** are widely used to evaluate semantic understanding of urban road scenes.

They provide annotated street-scene images containing classes relevant to autonomous driving and computer vision research.

---

## ⚠️ The Real Challenge

Segmentation becomes significantly more difficult when we leave controlled datasets and enter the real world.

Real-world autonomous driving environments introduce challenges such as:

* 🌙 Changing illumination
* 🌧️ Rain, fog and snow
* 🚗 Occlusion
* 📏 Large differences in object scale
* 🔲 Ambiguous object boundaries
* 🌃 Night-time driving
* ⚡ Strict real-time computing requirements

A useful automotive segmentation model needs to be:

**Accurate + Robust + Computationally Efficient**

---

## 🚗 From Pixels to Scene Understanding

The goal isn't simply to classify an image.

The goal is to transform **raw sensor data into a meaningful representation of the environment** that other systems can use to make decisions.

The progression is:

```text
Raw Camera Image
       ↓
Pixel-Level Classification
       ↓
Semantic Scene Understanding
       ↓
Environment Representation
       ↓
Decision-Making
```

Moving from:

> **"What objects are there?"**

to:

> **"What exactly is happening around the vehicle?"**

That's the power of **pixel-level scene understanding**.

---

## 🕹️ CARLA Demonstration

To demonstrate these concepts, I developed a solution using **CARLA Simulator** to show how image segmentation works in an autonomous-driving environment.

The project demonstrates how camera data can be processed to generate a **pixel-level semantic representation of the road scene**.

This provides a practical way to visualize the connection between:

```text
Camera Sensor
      ↓
Image
      ↓
Segmentation Model
      ↓
Pixel Classification
      ↓
Semantic Map
```

---

## 🧠 Key Takeaways

* **Semantic segmentation** assigns a class to every pixel.
* It provides richer spatial information than bounding-box-based object detection.
* **U-Net** uses an encoder-decoder architecture with skip connections.
* **IoU** is a common metric for evaluating segmentation performance.
* **Per-class IoU** helps identify weaknesses in specific object categories.
* Real-world deployment requires models to handle challenging environmental conditions.
* Automotive AI must balance **accuracy, robustness and computational efficiency**.
* Semantic segmentation is an important component of **environment perception and scene understanding**.

---

## 🛠️ Technologies

* 🐍 Python
* 🧠 TensorFlow
* 🚗 CARLA Simulator
* 👁️ Computer Vision
* 🤖 Deep Learning
* 🧩 U-Net
* 🛣️ Autonomous Driving

---

## 📚 Topics Covered

* Semantic Segmentation
* Computer Vision
* Convolutional Neural Networks
* U-Net
* Encoder-Decoder Architectures
* Skip Connections
* Intersection over Union
* Per-Class Evaluation
* Autonomous Driving Perception
* CARLA Simulation
* TensorFlow

---



In [4]:
import cv2
import carla
import random
import numpy as np

In [5]:
# ============================================================
# Connect to CARLA
# ============================================================

client = carla.Client("localhost", 2000)
client.set_timeout(20.0)
world = client.get_world()

print("Connected to Carla\n")
print(f"Map: {world.get_map().name}")



Connected to Carla

Map: Carla/Maps/Town10HD_Opt


In [6]:
# ============================================================
# Get blueprints and spawn vehicle
# ============================================================

blueprint_library = world.get_blueprint_library()
vehicle_bp = blueprint_library.find("vehicle.tesla.model3")
spawn_points = world.get_map().get_spawn_points()

ego_vehicle = world.try_spawn_actor(vehicle_bp, spawn_points[0])

print(f"Vehicle spawned {ego_vehicle.id}")

Vehicle spawned 24


In [7]:
# ============================================================
# Get blueprints and spawn points
# ============================================================

blueprint_library = world.get_blueprint_library()

vehicle_blueprints = list(
    blueprint_library.filter("vehicle.*")
)
spawn_points = world.get_map().get_spawn_points()

print(f"Available vehicle blueprints: {len(vehicle_blueprints)}")
print(f"Available spawn points: {len(spawn_points)}")


# ============================================================
# Randomize
# ============================================================
NUMBER_OF_VEHICLES = 50
TRAFFIC_MANAGER_PORT = 8000
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

random.shuffle(spawn_points)
random.shuffle(vehicle_blueprints)


# ============================================================
# Spawn vehicles
# ============================================================

vehicles = []

number_to_spawn = min(
    NUMBER_OF_VEHICLES,
    len(spawn_points)
)

print(f"Spawning {number_to_spawn} vehicles...")

for i in range(number_to_spawn):

    vehicle_bp = random.choice(vehicle_blueprints)
    spawn_point = spawn_points[i]

    vehicle = world.try_spawn_actor(
        vehicle_bp,
        spawn_point
    )

    if vehicle is not None:
        vehicles.append(vehicle)

        print(
            f"Spawned vehicle {vehicle.id}: "
            f"{vehicle.type_id}"
        )


print(f"\nSuccessfully spawned {len(vehicles)} vehicles.")


# ============================================================
# Configure Traffic Manager
# ============================================================

traffic_manager = client.get_trafficmanager(
    TRAFFIC_MANAGER_PORT
)

traffic_manager.set_global_distance_to_leading_vehicle(3.0)
traffic_manager.global_percentage_speed_difference(10.0)


# ============================================================
# Enable Autopilot
# ============================================================

print("\nEnabling autopilot...")

for vehicle in vehicles:

    vehicle.set_autopilot(
        True,
        TRAFFIC_MANAGER_PORT
    )


print("Traffic simulation started.")
print("Press Ctrl+C to stop.")


Available vehicle blueprints: 41
Available spawn points: 155
Spawning 50 vehicles...
Spawned vehicle 25: vehicle.micro.microlino
Spawned vehicle 26: vehicle.diamondback.century
Spawned vehicle 27: vehicle.mitsubishi.fusorosa
Spawned vehicle 28: vehicle.nissan.patrol
Spawned vehicle 29: vehicle.carlamotors.carlacola
Spawned vehicle 30: vehicle.nissan.patrol
Spawned vehicle 31: vehicle.dodge.charger_2020
Spawned vehicle 32: vehicle.tesla.model3
Spawned vehicle 33: vehicle.ford.crown
Spawned vehicle 34: vehicle.harley-davidson.low_rider
Spawned vehicle 35: vehicle.nissan.patrol_2021
Spawned vehicle 36: vehicle.vespa.zx125
Spawned vehicle 37: vehicle.ford.ambulance
Spawned vehicle 38: vehicle.mini.cooper_s_2021
Spawned vehicle 39: vehicle.ford.crown
Spawned vehicle 40: vehicle.ford.ambulance
Spawned vehicle 41: vehicle.volkswagen.t2
Spawned vehicle 42: vehicle.harley-davidson.low_rider
Spawned vehicle 43: vehicle.mercedes.sprinter
Spawned vehicle 44: vehicle.seat.leon
Spawned vehicle 45: v

In [7]:
sensors = blueprint_library.filter("sensor.*")

for sensor in sensors:
    print(sensor.id)

sensor.other.collision
sensor.camera.depth
sensor.camera.optical_flow
sensor.camera.normals
sensor.other.lane_invasion
sensor.camera.dvs
sensor.other.imu
sensor.other.gnss
sensor.other.obstacle
sensor.other.radar
sensor.lidar.ray_cast_semantic
sensor.lidar.ray_cast
sensor.camera.rgb
sensor.camera.semantic_segmentation
sensor.other.rss
sensor.camera.instance_segmentation
sensor.camera.cosmos_visualization
sensor.other.v2x
sensor.other.v2x_custom


In [6]:
vehicles = blueprint_library.filter("vehicle.*")
for vehicle in vehicles:
    print(vehicle.id)

vehicle.audi.a2
vehicle.nissan.micra
vehicle.audi.tt
vehicle.mercedes.coupe_2020
vehicle.bmw.grandtourer
vehicle.harley-davidson.low_rider
vehicle.ford.ambulance
vehicle.micro.microlino
vehicle.carlamotors.firetruck
vehicle.carlamotors.carlacola
vehicle.carlamotors.european_hgv
vehicle.ford.mustang
vehicle.chevrolet.impala
vehicle.lincoln.mkz_2020
vehicle.citroen.c3
vehicle.dodge.charger_police
vehicle.nissan.patrol
vehicle.jeep.wrangler_rubicon
vehicle.mini.cooper_s
vehicle.mercedes.coupe
vehicle.dodge.charger_2020
vehicle.ford.crown
vehicle.seat.leon
vehicle.toyota.prius
vehicle.yamaha.yzf
vehicle.kawasaki.ninja
vehicle.bh.crossbike
vehicle.mitsubishi.fusorosa
vehicle.tesla.model3
vehicle.gazelle.omafiets
vehicle.tesla.cybertruck
vehicle.diamondback.century
vehicle.mercedes.sprinter
vehicle.audi.etron
vehicle.volkswagen.t2
vehicle.lincoln.mkz_2017
vehicle.dodge.charger_police_2020
vehicle.vespa.zx125
vehicle.mini.cooper_s_2021
vehicle.nissan.patrol_2021
vehicle.volkswagen.t2_2021


In [8]:
# ============================================================
# Attach Semantic Segmentation Camera
# ============================================================

CAMERA_POS_Z = 3
CAMERA_POS_X = 0

camera_init_trans = carla.Transform(carla.Location(z = CAMERA_POS_Z, x = CAMERA_POS_X))
camera_bp = blueprint_library.find('sensor.camera.semantic_segmentation')
sem_camera = world.spawn_actor(camera_bp, camera_init_trans, attach_to=ego_vehicle)

In [9]:
# Camera Resolution

image_w = camera_bp.get_attribute("image_size_x").as_int()
image_h = camera_bp.get_attribute("image_size_y").as_int()



In [10]:
# ============================================================
# Camera Callback
# ============================================================

def sem_callback(image, data_dict):
    image.convert(carla.ColorConverter.CityScapesPalette)
    data_dict['sem_image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))


# Store semantic segmentation image
sensor_data = {
    'sem_image': np.zeros((image_w, image_h, 4))
}

 
# Start semantic segmentation camera
sem_camera.listen(lambda image: sem_callback(image, sensor_data))

In [18]:
# ============================================================
# Enable vehicle autopilot
# ============================================================

# traffic_manager = client.get_trafficmanager()
ego_vehicle.set_autopilot(True)

print("Autopilot enabled")
print("==============================")
print("Carla Semantic Segmentation running")
print("Press Q in the camera window to exit")
print("==============================")

Autopilot enabled
Carla Semantic Segmentation running
Press Q in the camera window to exit


In [ ]:
# Display semantic segmentation live
cv2.namedWindow("Semantic Segmentation", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Semantic Segmentation", 1920, 1080)
while True:


    cv2.imshow(
        'Semantic Segmentation',
        sensor_data['sem_image']
    )

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


# Stop camera
sem_camera.stop()

cv2.destroyAllWindows()

: 